In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(page_content="Hello, world!", metadata={"source": "example.txt","pages": 1,"author": "Yugank","date": "2026-06-03"})
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Yugank', 'date': '2026-06-03'}, page_content='Hello, world!')

In [4]:
## Create Directory for Text Files
import os
os.makedirs("../data/text_files", exist_ok=True)

In [5]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [14]:
## Textloader
### we can use the TextLoader class from langchain_community.document_loaders to load text files into Document objects. This allows us to easily work with the content of the text files in our applications.
from langchain_community.document_loaders import TextLoader
### we can use this also -> from langchain.document_loaders import TextLoader
loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
documents = loader.load()
print(documents[0].page_content)

Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.


In [24]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader
### we can use this also -> from langchain.document_loaders import DirectoryLoader
loader = DirectoryLoader(
    "../data/text_files", ## directory path
    glob="**/*.txt", ## pattern to match files
    loader_cls=TextLoader, ### loader class to use for loading files
    loader_kwargs={"encoding": "utf-8"}, ## additional arguments to pass to the loader class
    show_progress=False
) 
documents = loader.load()
documents
print(f"✅ Loaded {len(documents)} documents from the directory.")
print(documents[0].page_content + "\n")
print(documents[1].page_content)

✅ Loaded 2 documents from the directory.
Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards 

In [18]:
### for pdf
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader,PyMuPDFLoader
### we can use this also -> from langchain.document_loaders import PyPDFLoader,PyMuPDFLoader
dir_loader = DirectoryLoader(
    "../data/pdf", ## directory path
    glob="**/*.pdf", ## pattern to match files
    loader_cls=PyMuPDFLoader, ### loader class to use for loading files
    show_progress=False
)
pdf_documents = dir_loader.load()
pdf_documents
print(f"✅ Loaded {len(pdf_documents)} PDF documents from the directory." + "\n")
print(pdf_documents[0].page_content + "\n")
print(pdf_documents[1].page_content)

✅ Loaded 2 PDF documents from the directory.

🇮🇳 Independence Day 
📅 Date 
15 August 
📌 What is it? 
Independence Day is celebrated every year to mark India's independence from British rule 
in 1947. 
⭐ Key Points 
●​ India became independent on 15 August 1947. 
●​ Jawaharlal Nehru became the first Prime Minister of independent India. 
●​ The Prime Minister hoists the National Flag at the Red Fort. 
●​ The day honors the sacrifices of freedom fighters. 
●​ It represents freedom, unity, and patriotism.

🇮🇳 Republic Day 
📅 Date 
26 January 
📌 What is it? 
Republic Day is celebrated every year to commemorate the adoption of the Constitution of 
India, which came into effect on 26 January 1950. 
⭐ Key Points 
●​ India became a republic on 26 January 1950. 
●​ The Constitution of India came into effect on this day. 
●​ The President of India unfurls the National Flag. 
●​ A grand Republic Day Parade is held in New Delhi. 
●​ It represents democracy, equality, and constitutional values.


In [7]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [20]:
### for CSV
from langchain_community.document_loaders import CSVLoader
### we can use this also -> from langchain.document_loaders import CSVLoader
csv_loader = CSVLoader(
    file_path="../data/csv/sales.csv", ## path to the CSV file
    encoding="utf-8", ## encoding of the CSV file
    csv_args={"delimiter": ","}, ## additional arguments to pass to the CSV reader
)
csv_documents = csv_loader.load()
csv_documents
print(f"✅ Loaded {len(csv_documents)} CSV documents from the directory." + "\n")
print(csv_documents[0].page_content + "\n")
print(csv_documents[1].page_content)

✅ Loaded 10 CSV documents from the directory.

order_id: 1001
product: Laptop
category: Electronics
quantity: 2
price: 65000
city: Delhi
order_date: 2026-01-05

order_id: 1002
product: Mouse
category: Accessories
quantity: 5
price: 800
city: Mumbai
order_date: 2026-01-07


In [21]:
type(csv_documents[0])

langchain_core.documents.base.Document

EMBEDDING AND VECTOR STORE DB

In [6]:
import numpy as np 
from sentence_transformers import SentenceTransformer ## embedding model
import chromadb 
from chromadb.config import Settings 
import uuid ## for generating unique identifiers , because we will use it to generate unique IDs for our documents in the vector store 
from typing import List, Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity ## for calculating cosine similarity between vectors,while doing retrieval from vector db

In [4]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""    ### constructors are special methods that are called when an object is created from a class and allow the class to initialize the attributes of the class.
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):  ## this is for the model name of the sentence transformer model we want to use for generating embeddings
        """Initializes the EmbeddingManager with a specified model.
        Args:
            model_name (str): Hugging Face model name for the SentenceTransformer.
        """
        self.model_name = model_name ## intilialize the model name attribute with the provided model name
        self.model = None ## we intialize the model attribute to None, which will later hold the actual model instance
        self._load_model() ## Load the model during initialization
    
    def _load_model(self):
        """Loads the SentenceTransformer model."""
        try:
            print(f'Loading embedding model: {self.model_name}')
            self.model = SentenceTransformer(self.model_name) ## we load the model using the SentenceTransformer class and store it in the model attribute
            print(f"Model loaded successfully. Embedding dimensions : {self.model.get_embedding_dimension()}") ## 384 default
        except Exception as e:
            print(f"Error loading model: {self.model_name}. Exception: {e}")
            raise
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generates embeddings for a list of texts.
        Args:
            texts (List[str]): List of text strings to embed.
        Returns:
            np.ndarray: Array of embedding vectors.
        """
        if not self.model:
            raise ValueError("Model is not loaded. Please check the model name or loading process.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True) ## we generate embeddings for text 
        print(f"Embeddings generated successfully. Shape: {embeddings.shape}")
        return embeddings
    
    # def get_embedding_dimension(self) -> int:
    #     """Returns the dimension of the embeddings produced by the model."""
    #     if not self.model:
    #         raise ValueError("Model is not loaded. Please check the model name or loading process.")
    #     return self.model.get_embedding_dimension()
    
    ### initialize the embedding manager
    
embedding_manager = EmbeddingManager()  ## we create an instance of the EmbeddingManager class, specifying the model name to use for generating embeddings 
embedding_manager 
    

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12080.91it/s]


Model loaded successfully. Embedding dimensions : 384


VECTORE STORE

In [ ]:
import os

class VectorStore:
    """Handles storage and retrieval of document embeddings using ChromaDB."""
    def __init__(self, collection_name: str = "pdf_document", persist_directory: str = "../data/vector_store"):
        """Initializes the VectorStore with a specified collection name and persistence directory.
        Args:
            collection_name (str): Name of the ChromaDB collection.
            persist_directory (str): Directory to persist the vector store.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initializes the ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory) ## create a client for the ChromsDB
            
            #Get or create the collection. && where the vectors areto stored
            self.collection = self.client.get_or_create_collection(name=self.collection_name , metadata={"description":"PDF document embeddings for RAG"})
            print(f"Vector store initialized successfully. Collection: {self.collection_name}, Persist directory: {self.persist_directory}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store. Exception: {e}")
            raise
        
    ##for adding documents to the vector store
    def add_documents(self, documents: List[Any], embedding_manager: np.ndarray):
        """Adds documents to the vector store after generating embeddings.
        Args:
            documents : List of LangChain Documents
            embedding_manager (EmbeddingManager): Corresponding embedding for the documents.
        """
        if len(documents) != len(embedding_manager):
            raise ValueError("The number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store...")
        
        #preapre data for chromaDb
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []
        
        for i, (doc,embedding) in enumerate(zip(documents, embedding_manager)):
            #Generate Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"  ## we generate a unique ID for each document using the uuid library, which creates a random UUID and we take the first 8 characters of its hexadecimal representation
            ids.append(doc_id)
            
            #Prepare metadata
            metadata = dict(doc.metadata)  ## we create a copy of the document's metadata to avoid modifying the original metadata
            metadata['doc_index'] = i  ## we add an additional key-value pair to the metadata dictionary, indicating the index of the document in the list
            metadata['content_length'] = len(doc.page_content)  ## we add another key-value pair to the metadata dictionary, indicating the length of the document's content
            metadatas.append(metadata)
            
            #Documents content
            documents_texts.append(doc.page_content)
            
            #Embeddings
            embeddings_list.append(embedding_manager.tolist())  ## we convert the embedding from a numpy array to a list, as ChromaDB expects embeddings in list format
        
        #Add to ChromaDB collection    
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Documents added successfully. Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store. Exception: {e}")
            raise
            
        
vector_store = VectorStore()
vector_store

Error initializing vector store. Exception: name 'os' is not defined


NameError: name 'os' is not defined